In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score

def generate_forest_fire_data(samples=300):
    """Generates synthetic meteorological and environmental data for forest fire prediction."""
    np.random.seed(42)
    
    # Environmental Features
    temperature = np.random.uniform(10, 45, samples)         # °C
    relative_humidity = np.random.uniform(10, 90, samples)    # %
    wind_speed = np.random.uniform(2, 35, samples)            # km/h
    rainfall = np.random.uniform(0, 15, samples)              # mm
    fwi = np.random.uniform(0, 50, samples)                    # Fire Weather Index
    
    # Logic for fire occurrence probability
    risk_score = (
        (0.4 * temperature) 
        - (0.3 * relative_humidity) 
        + (0.2 * wind_speed) 
        - (0.5 * rainfall) 
        + (0.3 * fwi)
    )
    
    # Fire Class: 1 = Fire, 0 = No Fire
    fire_occurred = (risk_score > np.median(risk_score)).astype(int)
    
    # Burned Area (hectares): Only applies when fire occurs
    burned_area = np.where(
        fire_occurred == 1,
        np.maximum(0, (risk_score * 0.5) + np.random.normal(0, 5, samples)),
        0.0
    )
    
    df = pd.DataFrame({
        'Temperature_C': temperature,
        'Relative_Humidity_Pct': relative_humidity,
        'Wind_Speed_kmh': wind_speed,
        'Rainfall_mm': rainfall,
        'FWI_Index': fwi,
        'Fire_Occurred': fire_occurred,
        'Burned_Area_ha': burned_area
    })
    return df

def train_and_evaluate_models(df):
    """Trains Random Forest models for both Classification (Fire Risk) and Regression (Area Burned)."""
    X = df[['Temperature_C', 'Relative_Humidity_Pct', 'Wind_Speed_kmh', 'Rainfall_mm', 'FWI_Index']]
    y_class = df['Fire_Occurred']
    y_reg = df['Burned_Area_ha']
    
    # Split Data
    X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
        X, y_class, y_reg, test_size=0.2, random_state=42
    )
    
    # 1. Random Forest Classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, yc_train)
    yc_pred = clf.predict(X_test)
    
    accuracy = accuracy_score(yc_test, yc_pred)
    f1 = f1_score(yc_test, yc_pred)
    
    # 2. Random Forest Regressor
    reg = RandomForestRegressor(n_estimators=100, random_state=42)
    reg.fit(X_train, yr_train)
    yr_pred = reg.predict(X_test)
    
    r2 = r2_score(yr_test, yr_pred)
    rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
    
    return clf, reg, accuracy, f1, r2, rmse

def predict_fire_risk(clf, reg, new_conditions):
    """Predicts fire occurrence and estimated burned area for a new weather condition."""
    input_df = pd.DataFrame([new_conditions])
    
    fire_pred = clf.predict(input_df)[0]
    fire_prob = clf.predict_proba(input_df)[0][1]
    
    if fire_pred == 1:
        estimated_area = round(reg.predict(input_df)[0], 2)
    else:
        estimated_area = 0.0
        
    return fire_pred, fire_prob, estimated_area

# --- Main Execution Block ---
if __name__ == "__main__":
    # 1. Dataset Generation
    data = generate_forest_fire_data(samples=300)
    print("--- Historical Meteorological Dataset (First 5 Rows) ---")
    print(data.head().round(2).to_string(index=False))
    print("\n" + "="*60 + "\n")
    
    # 2. Model Training & Evaluation
    clf, reg, accuracy, f1, r2, rmse = train_and_evaluate_models(data)
    
    print("--- Model Performance Metrics ---")
    print(f"[Classifier] Accuracy Score       : {accuracy * 100:.2f}%")
    print(f"[Classifier] F1 Score             : {f1:.4f}")
    print(f"[Regressor]  R² Score (Burned Area): {r2:.4f}")
    print(f"[Regressor]  RMSE (Burned Area)   : {rmse:.2f} hectares")
    print("\n" + "="*60 + "\n")
    
    # 3. Feature Importance
    print("--- Feature Importance (Classifier) ---")
    features = ['Temperature_C', 'Relative_Humidity_Pct', 'Wind_Speed_kmh', 'Rainfall_mm', 'FWI_Index']
    for feat, imp in zip(features, clf.feature_importances_):
        print(f"Feature [{feat:22s}] Importance: {imp * 100:.2f}%")
    print("\n" + "="*60 + "\n")
    
    # 4. New Scenario Prediction
    weather_scenario = {
        'Temperature_C': 38.5,
        'Relative_Humidity_Pct': 18.0,
        'Wind_Speed_kmh': 28.0,
        'Rainfall_mm': 0.0,
        'FWI_Index': 35.2
    }
    
    fire_pred, fire_prob, estimated_area = predict_fire_risk(clf, reg, weather_scenario)
    
    print("--- Real-time Environmental Scenario ---")
    for key, val in weather_scenario.items():
        print(f"{key:24s}: {val}")
    
    print("-" * 60)
    status = "CRITICAL RISK (Fire Expected)" if fire_pred == 1 else "LOW RISK (No Fire)"
    print(f"Fire Prediction Status      : {status}")
    print(f"Fire Probability            : {fire_prob * 100:.1f}%")
    print(f"Estimated Burned Area       : {estimated_area} hectares")

--- Historical Meteorological Dataset (First 5 Rows) ---
 Temperature_C  Relative_Humidity_Pct  Wind_Speed_kmh  Rainfall_mm  FWI_Index  Fire_Occurred  Burned_Area_ha
         23.11                  14.13            7.57         3.12      37.91              1            0.00
         43.28                  52.51           11.19         0.40       1.23              1           13.44
         35.62                  53.25            7.84         2.72       1.11              0            0.00
         30.95                  60.99            4.93         8.75      16.18              0            0.00
         15.46                  68.09            5.98         6.32      24.43              0            0.00


--- Model Performance Metrics ---
[Classifier] Accuracy Score       : 86.67%
[Classifier] F1 Score             : 0.8750
[Regressor]  R² Score (Burned Area): 0.6086
[Regressor]  RMSE (Burned Area)   : 2.80 hectares


--- Feature Importance (Classifier) ---
Feature [Temperature_C         